In [2]:
import os
from dotenv import load_dotenv

In [15]:
load_dotenv()

True

In [5]:
from langchain_openai import ChatOpenAI

In [6]:
llm = ChatOpenAI(model= 'gpt-5.4-mini', temperature=0)

In [7]:
llm.invoke("what is deep agents")

AIMessage(content='“Deep agents” usually refers to **AI agents that can reason and act over multiple steps**, often with more persistence and autonomy than a simple chatbot.\n\nIn plain English, a deep agent typically can:\n\n- **Plan ahead** instead of answering in one shot\n- **Use tools** like web search, code execution, databases, or APIs\n- **Remember context** across steps in a task\n- **Break big goals into smaller sub-tasks**\n- **Reflect and adjust** when something doesn’t work\n\n### Example\nIf you ask:\n\n> “Research the best laptops under $1,500, compare them, and draft a recommendation.”\n\nA deep agent might:\n1. Search the web\n2. Gather specs and prices\n3. Compare options\n4. Summarize tradeoffs\n5. Produce a final recommendation\n\n### Why “deep”?\nThe term often suggests the agent uses **deeper reasoning, longer-horizon planning, or layered decision-making** rather than just generating text immediately.\n\n### Important note\n“Deep agents” is not always a strict tec

In [16]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [1]:
from tavily import TavilyClient

In [11]:
tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [12]:
def internet_search(query: str, max_results: int = 5):
    """Search the web for current information on topic. Use for any question
    that needs up-to-date facts, trends or competitor/market information."""
    return tavily.search(query=query, max_results=max_results)

    

In [13]:
from deepagents import create_deep_agent

In [14]:
research_agent = create_deep_agent(
    tools = [internet_search],
    system_prompt= (
        "You are an expert researcher. Always: (1) Use write_todos to plan,"
        "(2) use internet_search to gather facts, (3) save findings to file,"
        "(4) write a clear final report"
    ),
    model = llm
)

In [15]:
out = research_agent.invoke({
    "messages": [{"role": "user",
                  "content": "Research the latest news from apple as of June 9th 2026"}]
})

In [19]:
print(out["messages"][-1].content)

Apple’s latest news as of June 9, 2026 is dominated by WWDC 2026 and its Siri AI launch.

Key takeaways:
- Apple unveiled a major Siri overhaul branded **Siri AI** at WWDC on June 8.
- Apple expanded **Apple Intelligence** across more apps and system experiences.
- Apple announced **new child safety / parental control features**.
- Apple added **new developer intelligence frameworks and Xcode tools**.
- The main follow-up story on June 9 was **regulatory friction in the EU** around Siri AI and Apple’s compliance with tech rules.
- CNBC also reported that some Siri AI features may face **regional rollout limits** in the EU and China.

Most important sources:
- **Apple Newsroom** — official WWDC announcements
- **Reuters** — June 8–9 coverage of Siri AI and EU regulatory issues
- **CNBC** — WWDC live coverage and rollout analysis

Saved findings to:
`/apple_news_research_2026-06-09.md`

If you want, I can turn this into a one-page briefing with links and a timeline.


In [20]:
llm.invoke("LAtest news from Apple as of June9th 2026")

AIMessage(content='I can help, but I don’t have live news access in this chat, and my knowledge only goes up to **2024-06**. So I can’t reliably tell you the latest Apple news **as of June 9, 2026** without a web source.\n\nIf you want, I can do one of these:\n\n1. **Summarize Apple’s most recent major news up to 2024-06**\n2. **Help you find the latest June 9, 2026 Apple news** by telling you the best places to check\n3. **Analyze any Apple article or press release you paste here** and summarize it quickly\n\nGood places to check for the latest Apple news:\n- Apple Newsroom: https://www.apple.com/newsroom/\n- Apple Investor Relations: https://investor.apple.com/\n- Reuters / Bloomberg / The Verge / MacRumors / 9to5Mac\n\nIf you want, I can also give you a **one-page “what to look for in Apple news” checklist** or a **quick summary of Apple’s latest known product and strategy themes**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 230, '

In [21]:
for chunk in research_agent.stream(
    {"messages" : [{"role" :"user", "content": "Latest Job Posting on AI Engineer as of June 9th 2026"}]},
    stream_mode = "values",
):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

Latest Job Posting on AI Engineer as of June 9th 2026
================================== Ai Message ==================================
Tool Calls:
  write_todos (call_TrNZsE81ThEqHhacf1fCGPtM)
 Call ID: call_TrNZsE81ThEqHhacf1fCGPtM
  Args:
    todos: [{'content': 'Search the web for current AI Engineer job postings and identify the most recent relevant posting as of June 9, 2026', 'status': 'in_progress'}, {'content': 'Save findings to a file', 'status': 'pending'}, {'content': 'Provide a concise final report with the posting details and source', 'status': 'pending'}]
================================= Tool Message =================================
Name: write_todos

Updated todo list to [{'content': 'Search the web for current AI Engineer job postings and identify the most recent relevant posting as of June 9, 2026', 'status': 'in_progress'}, {'content': 'Save findings to a file', 'status': 'pending'}, {

In [24]:
research_sub = {
    "name": "research-agent",
    "description": "finds seed keywords by researching the topic. Use FIRST",
    "system_prompt": (
        "You are a keyword research specialist. Use internet_seach to find"
        "10-15 high value SEED Keywords. Save them to 'seeds.md'."
    ),
    "tools": [internet_search]
}

In [25]:
def expand_keywords(seed: str):
    """ Expand a seed keyword into long-tail variations using common modifiers.
    use this when you need keyword ideas for seo or google ads """
    modifiers = ["course", "training", "online", "certification", "for beginners",
                 "with placements", "professionals","2026"]
    return [f"{seed} {m}" for m in modifiers]

In [26]:
def classify_intent(keyword: str):
    """Classify the intent of a keywords search as commercial, informational, or navigational, and
    assign an ad priority. use to triage keyword list"""
    kw = keyword.lower()
    if any(w in kw for w in ["free","salary", "what is", "vs","meaning"]):
        return {"keyword": keyword, "intent": "informational", "ad_priority": "low"}
    elif any(w in kw for w in ["buy", "purchase", "for sale", "discount", "deal"]):
        return {"keyword": keyword, "intent": "commercial", "ad_priority": "high"}
    else:
        return {"keyword": keyword, "intent": "navigational", "ad_priority": "medium"}

In [27]:
research_sub = {
    "name": "research-agent",
    "description": "finds seed keywords by researching the topic. Use FIRST",
    "system_prompt": (
        "You are a keyword research specialist. Use internet_seach to find"
        "10-15 high value SEED Keywords. Save them to 'seeds.md'."
    ),
    "tools": [internet_search]
}

expansion_sub = {
    "name": "expansion-agent",
    "description": "expands seed keywords into long-tail variations. Use SECOND",
    "system_prompt": (
        "Read 'seeds.md' for each seed call expand_keywords. deduplicate"
        "and save everything to 'expanded.md'."
    ),
    "tools": [expand_keywords]
}

classify_sub = {
    "name": "classify-agent",
    "description": "classifies keywords by intent and ad priority and flag negatives. Use THIRD",
    "system_prompt": (
        "Read 'expanded.md' for each keyword call classify_intent. deduplicate"
        "Group into High Priority, Commercial, informational and negative keywords and save to final.md"
    ),
    "tools": [classify_intent]
}

supervisor = create_deep_agent(
    tools = [internet_search, expand_keywords, classify_intent],
    system_prompt= (
        "You are the Supervisor of a Google Ads Team in BEPEC. Use write_todos to plan"
        "then delegate in order: research_agent, expansion_agent, classify_agent"
        "never do the work yourself. finally read final.md and summarise the top 10"
        "commerical keywords plus a negative keyword list"
    ),
    model = llm,
    subagents= [research_sub, expansion_sub, classify_sub]
)

In [28]:
out = supervisor.invoke({
    "messages": [{"role": "user",
                  "content": "Find google ads keywords for AI Training for Working Professionals from India, UK, USA"}]

})

In [30]:
print(out['messages'][-1].content)

Top 10 commercial Google Ads keywords:

1. AI training for professionals certification  
2. AI course for working professionals certification  
3. artificial intelligence course for professionals certification  
4. AI certification course training  
5. AI certificate for professionals training  
6. professional AI certification training  
7. AI training program certification  
8. AI upskilling course certification  
9. AI career training certification  
10. job-ready AI skills certification  

Best negative keyword list:
- free
- pdf
- ebook
- ppt
- slides
- tutorial
- definition
- meaning
- what is
- examples
- jobs
- job openings
- salary
- internship
- school
- college
- university
- kids
- children
- game
- beginner
- beginners
- 2026
- wiki
- research
- history
- notes
- assignment
- question paper

Recommended high-intent modifiers to keep testing:
- certification
- course
- training
- online
- with placements
- professional
- job-ready
- corporate

If you want, I can also turn t